# System Design I: Software Systems

Notebook for **Chapter 8** of the *career* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/career/ch08.html

Tip: change a number, predict the output, then run the cell again.

## 8.1 The Design Framework and the Words You Need

## 8.2 Estimation With Little's Law

In [ ]:
new_per_month = 100_000_000
read_write = 100
secs_month = 30 * 86400
writes = new_per_month / secs_month
reads = writes * read_write
print("writes/s      :", round(writes, 1))
print("reads/s avg   :", round(reads))
print("reads/s peak  :", round(reads * 3))
records = new_per_month * 12 * 5
print("records 5y    :", f"{records:,}")
print("storage TB    :", records * 500 / 1e12)
print("7-char base62 :", f"{62**7:,}", "ratio", round(62**7 / records))
print("bandwidth MB/s:", round(reads * 500 / 1e6, 2))

writes/s      : 38.6
reads/s avg   : 3858
reads/s peak  : 11574
records 5y    : 6,000,000,000
storage TB    : 3.0
7-char base62 : 3,521,614,606,208 ratio 587
bandwidth MB/s: 1.93


## 8.3 Caching

In [ ]:
reads = 3858
for h in (0.0, 0.5, 0.9, 0.99):
    avg_ms = h * 1 + (1 - h) * 10
    db = reads * (1 - h)
    print(f"hit {h:4.2f}: avg {avg_ms:5.2f} ms, DB reads/s {db:7.1f}")
hot = 100_000_000 * 0.2 * 500 / 1e9
print("cache GB for hottest 20% of 100M urls:", hot)

hit 0.00: avg 10.00 ms, DB reads/s  3858.0
hit 0.50: avg  5.50 ms, DB reads/s  1929.0
hit 0.90: avg  1.90 ms, DB reads/s   385.8
hit 0.99: avg  1.09 ms, DB reads/s    38.6
cache GB for hottest 20% of 100M urls: 10.0


## 8.4 Sharding, Replication and Consistent Hashing

In [ ]:
import hashlib, bisect
def h(s): return int(hashlib.md5(s.encode()).hexdigest(), 16)
keys = [f"key{i}" for i in range(100000)]
moved = sum(1 for k in keys if h(k) % 4 != h(k) % 5)
print("mod 4 -> mod 5 moved:", moved / len(keys))
def ring(nodes, vn=100):
    pts = sorted((h(f"{n}#{v}"), n) for n in nodes for v in range(vn))
    return [p for p, _ in pts], [n for _, n in pts]
def owner(r, k):
    ps, ns = r
    i = bisect.bisect(ps, h(k)) % len(ps)
    return ns[i]
r4 = ring(["A","B","C","D"]); r5 = ring(["A","B","C","D","E"])
moved = sum(1 for k in keys if owner(r4, k) != owner(r5, k))
print("ring 4 -> 5 moved   :", moved / len(keys))
from collections import Counter
print(sorted(Counter(owner(r5, k) for k in keys).items()))

mod 4 -> mod 5 moved: 0.80045
ring 4 -> 5 moved   : 0.19343
[('A', 20388), ('B', 22592), ('C', 19712), ('D', 17965), ('E', 19343)]


## 8.5 Queues, Retries and Reliability

In [ ]:
jobs = 1000; svc = 0.020
busy = jobs * svc
print("busy workers (Little):", busy)
for u in (0.5, 0.6, 0.8, 0.9, 0.95):
    print(f"util {u:.2f} -> workers {busy/u:5.1f}, time in system = service x {1/(1-u):4.1f} = {svc/(1-u)*1000:6.1f} ms")

busy workers (Little): 20.0
util 0.50 -> workers  40.0, time in system = service x  2.0 =   40.0 ms
util 0.60 -> workers  33.3, time in system = service x  2.5 =   50.0 ms
util 0.80 -> workers  25.0, time in system = service x  5.0 =  100.0 ms
util 0.90 -> workers  22.2, time in system = service x 10.0 =  200.0 ms
util 0.95 -> workers  21.1, time in system = service x 20.0 =  400.0 ms


In [ ]:
for a in (0.99, 0.999, 0.9999):
    print(a, "downtime min/month:", round((1-a) * 30 * 1440, 1))
print("3 services in series at 99.9% each:", round(0.999**3, 6))
print("2 replicas each 99%, parallel:", 1 - 0.01**2)
base, cap = 0.1, 5
print([min(cap, base * 2**i) for i in range(8)])

0.99 downtime min/month: 432.0
0.999 downtime min/month: 43.2
0.9999 downtime min/month: 4.3
3 services in series at 99.9% each: 0.997003
2 replicas each 99%, parallel: 0.9999
[0.1, 0.2, 0.4, 0.8, 1.6, 3.2, 5, 5]


## 8.6 Worked Design 1: URL Shortener

## 8.7 Worked Design 2: Rate Limiter

In [ ]:
class TokenBucket:
    def __init__(self, rate, capacity):
        self.rate, self.capacity = rate, capacity
        self.tokens, self.last = capacity, 0.0
    def allow(self, now):
        self.tokens = min(self.capacity, self.tokens + (now - self.last) * self.rate)
        self.last = now
        if self.tokens >= 1:
            self.tokens -= 1
            return True
        return False

b = TokenBucket(rate=5, capacity=10)
out = [b.allow(0.0) for _ in range(15)]
print("burst of 15 at t=0:", "".join("Y" if x else "N" for x in out))
out = [b.allow(1.0 + i * 0.125) for i in range(16)]
print("8 req/s for 2 s   :", "".join("Y" if x else "N" for x in out))

burst of 15 at t=0: YYYYYYYYYYNNNNN
8 req/s for 2 s   : YYYYYYYYYYYNYYNY


## 8.8 Worked Design 3: News Feed

In [ ]:
dau = 50_000_000
opens = 10
print("feed reads/s avg :", round(dau * opens / 86400))
print("feed reads/s peak:", round(dau * opens / 86400 * 3))
posts = dau * 0.2 / 86400
print("posts/s          :", round(posts, 1))
fan = 200
print("fanout writes/s  :", round(posts * fan))
print("entries/day      :", dau * 0.2 * fan, "-> GB at 16 B:", dau * 0.2 * fan * 16 / 1e9)
print("one celebrity post to 10M followers:", 10_000_000, "writes")
print("at 20k writes/s that takes", 10_000_000 / 20000, "s")

feed reads/s avg : 5787
feed reads/s peak: 17361
posts/s          : 115.7
fanout writes/s  : 23148
entries/day      : 2000000000.0 -> GB at 16 B: 32.0
one celebrity post to 10M followers: 10000000 writes
at 20k writes/s that takes 500.0 s


## Chapter recap

## Practice set

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/career/ch08.html